# 02 - Per-station positioning solutions

Reads cached RINEX from `data/cors/`, applies the v1 climatological positioning model (`gannon_analysis.positioning`), and produces per-station horizontal-error time series.

**Important methodology note.** This artifact uses a *climatological* positioning model in v1. Pseudo-range SPP from raw GPS observables, PPP refinement, and equipment-specific transfer functions are scheduled for v2. See `docs/methodology.md` for the full discussion.

In [ ]:
from pathlib import Path

from gannon_analysis.swpc import fetch_dst, fetch_kp, join_kp_to_minute_grid

data_dir = Path("../data")
swpc_cache = data_dir / "swpc"
swpc_cache.mkdir(parents=True, exist_ok=True)
kp = fetch_kp(cache_path=swpc_cache / "kp.txt")
dst = fetch_dst(cache_path=swpc_cache)
print(f"Kp rows: {len(kp)}; peak Kp: {kp['kp'].max():.1f}")
print(f"Dst rows: {len(dst)}; min Dst: {dst['dst'].min()} nT")

In [ ]:
kp_minute = join_kp_to_minute_grid(kp)
kp_minute.head()

## Solve SPP for one station

In [ ]:
from gannon_analysis.positioning import solve_spp
from gannon_analysis.stations import station_by_id

station = station_by_id("iaal")
rinex_day = data_dir / "cors" / "2024" / "131" / "iaal" / "iaal1310.24d.gz"
df = solve_spp(rinex_day, station, kp_series=kp_minute, dst_series=dst)
df.describe()[["h_error_2d_m", "kp", "dst_nT"]]

## Plot 2D error vs UTC for that station

In [ ]:
import matplotlib.pyplot as plt

from gannon_analysis.swpc import kp_severity_color

fig, ax = plt.subplots(figsize=(11, 4))
colors = [kp_severity_color(float(k)) for k in df["kp"][::30]]
ax.scatter(df.index[::30], df["h_error_2d_m"][::30], c=colors, s=3, alpha=0.7)
ax.axhline(0.025, color="#1a9850", ls="--", label="Planting (2.5 cm)")
ax.axhline(0.05, color="#f46d43", ls="--", label="Spraying (5 cm)")
ax.set_yscale("log")
ax.set_xlabel("UTC")
ax.set_ylabel("2D horizontal error (m, log)")
ax.set_title(f"{station.station_id.upper()} ({station.state}) - 2024-05-10 - color: Kp severity")
ax.legend()
ax.grid(True, alpha=0.4)
fig.autofmt_xdate()
plt.show()

## Run the whole corridor and write per-station figures

In [ ]:
from gannon_analysis.analysis import run_analysis
from gannon_analysis.plotting import plot_per_station_error_grid
from gannon_analysis.stations import stations_in_corridor

bundle = run_analysis(stations_in_corridor(), data_dir=data_dir)
out_dir = Path("../results/figures")
plot_per_station_error_grid(bundle, out_path=out_dir / "fig-02-per-station-grid.png")
print("Wrote per-station grid; see results/figures/fig-02-per-station-grid.png")

## Next: correlate with SWPC indices (notebook 03)